In [0]:
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv").option("cloudFiles.schemaLocation", "abfss://kesav@kesavfiles.dfs.core.windows.net/schema/divya/") \
    .load("abfss://kesav@kesavfiles.dfs.core.windows.net/input")

In [0]:
df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "abfss://kesav@kesavfiles.dfs.core.windows.net/checkpoint/divya") \
    .outputMode("append").option("path","abfss://kesav@kesavfiles.dfs.core.windows.net/oyput_tables_divya").\
        trigger(processingTime="30 seconds")   \
    .toTable("ram.default.divya")

In [0]:
%sql
describe history ram.default.divya

In [0]:
from pyspark.sql.functions import lit
folder_path = "abfss://divyacontainer@divyastorrage.dfs.core.windows.net/ram/"
archive_path = "abfss://divyacontainer@divyastorrage.dfs.core.windows.net/archivelk/"
files = dbutils.fs.ls(folder_path)
for i in files:
    print(i)
txt_files = sorted([f.path for f in files if f.path.endswith(".txt")], reverse=True)
print(txt_files)
latest_file = txt_files[0] if txt_files else None
print(latest_file)

update_dt = latest_file.split("_")[-1].replace(".txt/", "") if latest_file else None  # Extract date from filename
update_dt=update_dt.split(".")[0]

if latest_file:
    df = spark.read.format("csv").option('header', 'true').load([latest_file])
    df = df.withColumn("update_dt", lit(update_dt))

    # Write new file data to table in append mode
    df.write.mode("append").option("inferSchema", "true").option("mergeSchema", "true").saveAsTable("rama_table")

    display(df)

    # Move the file to archive
    dbutils.fs.mv(latest_file, archive_path + latest_file.split("/")[-1], recurse=True)
else:
    print("No .txt files found in the folder.")

In [0]:
jdbc_url = "jdbc:sqlserver://kesavserver.database.windows.net:1433;database=kesavdb"
table_name = "SalesLT.address"
properties = {
    "user": "kesav",
    "password": "Divya@444",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df = spark.read.format("jdbc").option("inferSchema", "true").option("mergeSchema", "true").option("url", jdbc_url).option("dbtable", table_name).options(**properties).load()


In [0]:
df.createOrReplaceTempView("df")

In [0]:
%sql
describe   df

In [0]:
data = [
    [1, "Alice", 50000],
    [2, "Bob", 60000],
    [3, "Charlie", 55000]
]

schema = ["id", "name", "salary"]

emp_df = spark.createDataFrame(data, schema)
display(emp_df)

In [0]:
emp_df.createOrReplaceTempView("emp_df")


In [0]:
data = [
    [1, "Alice"],
     [None, None],
]

schema = ["emP_id", "name"]

bckl_df = spark.createDataFrame(data, schema)
display(bckl_df)

In [0]:
bckl_df.createOrReplaceTempView("bckl_df")

In [0]:
%sql
select * from emp_df where id not in(select emP_id from bckl_df)

In [0]:
df = spark.read.format('csv').option('header','false').option("delimiter", "|,").load("abfss://kesav@kesavfiles.dfs.core.windows.net/rtp_paymnest.txt")

df.show()
df.write.mode('overwrite').option('header','false').format('delta').option('mergeSchema', 'true').option('overwriteSchema', 'true').saveAsTable("divya2.default.emdiv")


In [0]:
%sql
create external table emdiv (EmpID varchar(100), EmpName string,Department string, Salary varchar(100)) location 'abfss://kesav@kesavfiles.dfs.core.windows.net/rtp_payments/'

In [0]:
%sql
drop table emdiv

In [0]:
%sql
describe detail  emdiv;

In [0]:
%sql
describe emdiv;

In [0]:
%sql
create external table emdiv_pub (id varchar(100), name string, dept string, Salary varchar(100)) location 'abfss://kesav@kesavfiles.dfs.core.windows.net/rtp_payments_pub/'


In [0]:
%sql
select * from divya2.default.emdiv